# RLM Log Analysis Functions

This notebook provides utility functions to extract key data from RLM log files:
- **Final answer**: The agent's concluding response
- **Code blocks**: All code executed during the session
- **RLM calls**: Sub-LLM calls made via `llm_query()` / `llm_query_batched()`

In [1]:
import sys
import importlib


sys.path.append('/home/winnieyangwn/rlm/analysis')
import rlm_log_utils
importlib.reload(rlm_log_utils)
from rlm_log_utils import *

## Usage Example

Load the log file and extract key information:

# Load

In [10]:

LOG_PATH = "/checkpoint/maui_sft/winnieyangwn/rlm_dumps/errors_codebase/gpt-5_common_invalid_errors_codebase_503_2026-02-12_09-56-33_586ea944.jsonl"
# Load the log - first entry is metadata, rest are iterations
entries = load_rlm_log(LOG_PATH)
metadata = entries[0]
iterations = entries[1:]

print(f"Loaded {len(iterations)} iterations")

Loaded 7 iterations


# Metadata

In [11]:
# View metadata
print("=== METADATA ===")
for k, v in metadata.items():
    if k != "backend_kwargs":
        print(f"{k}: {v}")

=== METADATA ===
type: metadata
timestamp: 2026-02-12T09:56:33.727850
root_model: gpt-5
max_depth: 2
max_iterations: 100
backend: azure_openai
environment_type: local
environment_kwargs: {'setup_code': '\nimport pandas as pd\nfrom pathlib import Path\n\n# Load rollout data as DataFrame\nrollout_df = pd.read_json(\'/checkpoint/agentic-models/winnieyangwn/amaia_dumps/503/trajectories/503_metadata.jsonl\', lines=True)\n\n# Load codebase into dict\ncodebase = {}\ncodebase_root = Path(\'/checkpoint/agentic-models/winnieyangwn/amaia_dumps/503/code/2026_02_06_09_33_54\')\nfor ext in [\'.py\', \'.md\', \'.yaml\']:\n    for path in codebase_root.rglob(f\'*{ext}\'):\n        try:\n            rel_path = str(path.relative_to(codebase_root))\n            codebase[rel_path] = path.read_text(errors=\'ignore\')\n        except Exception:\n            pass  # Skip unreadable files\n\n# Load config YAML\nconfig_yaml = \'\'\'# python -m launchers.stool run name="503" script=apps.sea.eval config=apps/sea

In [12]:

# Compare with timestamp-based runtime
runtime = get_total_runtime(entries)
print(f"Timestamp-based runtime: {runtime.total_seconds():.2f}s")

Timestamp-based runtime: 366.49s


# Final Answer

In [13]:
# Get the final answer
final_answer = get_final_answer(iterations)
print("=== FINAL ANSWER ===")
print(final_answer if final_answer else "No final answer found")
# print(f"\n(Total length: {len(final_answer) if final_answer else 0} chars)")

=== FINAL ANSWER ===
Top 5 most common evaluation error messages among invalid submissions:
- FileNotFoundError: [Errno 2] No such file or directory: '/<path> — 699 occurrences
- Empty error output for failed submission. — 567 occurrences
- (empty) — 287 occurrences
- Validation error: Submission invalid! The attempt to grade the submission has resulted in the following error message: — 239 occurrences
- submission.csv not in solution — 111 occurrences

Root cause analysis and specific improvements per error type:

1) FileNotFoundError: [Errno 2] No such file or directory: '/<path>
Likely causes (based on agent code and traces):
- Hard-coded dataset paths/filenames that do not exist in the MLE Bench container. Common missing files observed include:
  • /root/data/train.csv — 68 occurrences
  • /root/data/Train.csv — 29 occurrences
  • /root/data/train.txt — 49 occurrences
  • /root/data/en_test.csv — 42 occurrences
  • /root/data/ru_test.csv — 32 occurrences
  • /root/data/train.zip — 